# Fase 2: Exploración y preprocesamiento del dataset

Este notebook implementa la exploración, limpieza, transformación y validación inicial del conjunto Active Street-Use Permits. La unidad de análisis se conserva como registro territorial del permiso; por ello, no se eliminan filas solo porque compartan `permit_number`.

## 1. Objetivo y decisiones de procesamiento

- Mantener una copia inmutable del archivo fuente.
- Identificar valores faltantes, duplicados exactos y tipos de datos.
- Convertir fechas y coordenadas a tipos analíticos.
- Calcular la duración solo cuando existen fechas válidas y coherentes.
- Conservar los faltantes de variables no esenciales y documentarlos como `No informado` en variables categóricas seleccionadas.
- Exportar una tabla procesada y un resumen de calidad para las fases posteriores.

In [21]:
# Importamos herramientas para trabajar con rutas.
from pathlib import Path

# Importamos NumPy para operaciones numéricas y Pandas para manipular tablas.
import numpy as np
import pandas as pd

In [22]:
# Indicamos la ruta relativa desde F2/notebooks hasta la raíz del repositorio.
archivo = '../../Active_Street-Use_Permits_20260909.csv'

# Creamos la carpeta donde se guardarán los resultados procesados.
PROCESSED_DIR = Path('../../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [23]:
# Cargamos el archivo CSV en un DataFrame llamado df.
df = pd.read_csv(archivo)

# Conservamos una copia original y otra de trabajo para el procesamiento.
raw_permits = df.copy()
permits = df.copy()

# Mostramos el tamaño de la tabla para comprobar la carga inicial.
print(f'Filas: {permits.shape[0]:,}')
print(f'Columnas: {permits.shape[1]}')

Filas: 6,852
Columnas: 29


In [24]:
# Verificamos visualmente las primeras filas del DataFrame.
print('Primeras filas del DataFrame:')
display(permits.head())

Primeras filas del DataFrame:


,permit_number,streetname,Cross Street 1,Cross Street 2,Permit Type,Agent,AgentPhone,Permit Purpose,Approved Date,Status,...,Y,Latitude,Longitude,bpa,Location,the_geom,analysis_neighborhood,supervisor_district,data_as_of,data_loaded_at
0,26EXC-01641,08TH ST,NATOMA ST,HOWARD ST,Excavation,Ronan Construction Inc.,415-779-5262,"8th St, Clay St and Leavenworth St - Pavement ...",04/15/2026,APPROVED,...,"2110825,76329","37,77644115815","-122,4118820325",NaN,POINT (-122.41188203249555 37.77644115815177),POINT (-122.411882032 37.776441158),South of Market,6.0,2026/04/16 03:34:45 AM,2026/04/17 05:23:32 AM
1,25EXC-01889,SAN BRUNO AVE,MANSELL ST \ SALINAS AVE,ORDWAY ST,Excavation,"Cratus, Inc.",(415)939-2840,WD-2876 8-Inch Ductile Iron Water Main Repl...,07/18/2025,ACTIVE,...,"2090569,998","37,7209782871","-122,40087332838",NaN,POINT (-122.40087332838284 37.720978287099676),POINT (-122.400873328 37.720978287),Portola,10.0,2025/09/10 05:11:34 AM,2025/09/12 05:08:57 AM
2,23IE-00344,WALNUT ST,PACIFIC AVE,JACKSON ST,StrtImprov,"Partner's Contracting, Inc.",415-492-9233,Remove and reconstruct sidewalk and existing (...,08/26/2025,APPROVED,...,"2116443,35732","37,79125689282","-122,44945871865",202210184608,POINT (-122.44945871865453 37.79125689282032),POINT (-122.449458719 37.791256893),Presidio Heights,2.0,2025/08/28 03:42:52 AM,2025/08/29 05:13:53 AM
3,25EXC-01889,CAMPBELL AVE,SAN BRUNO AVE,BRUSSELS ST,Excavation,"Cratus, Inc.",(415)939-2840,WD-2876 8-Inch Ductile Iron Water Main Repl...,07/18/2025,ACTIVE,...,"2088480,85453","37,71525239967","-122,40007624052",NaN,POINT (-122.40007624051567 37.715252399671314),POINT (-122.400076241 37.7152524),Visitacion Valley,10.0,2025/09/10 05:11:34 AM,2025/09/12 05:08:57 AM
4,25EXC-01889,CANDLESTICK COVE WAY,EXECUTIVE PARK BLVD,NaN,Excavation,"Cratus, Inc.",(415)939-2840,WD-2876 8-Inch Ductile Iron Water Main Repl...,07/18/2025,ACTIVE,...,"2087023,572266","37,71134702909","-122,39401110729",NaN,POINT (-122.39401110729395 37.71134702909027),POINT (-122.394011107 37.711347029),Bayview Hunters Point,10.0,2025/09/10 05:11:34 AM,2025/09/12 05:08:57 AM


In [25]:
# Revisamos los nombres de todas las columnas disponibles.
print('Nombres de las columnas:')
print(permits.columns.tolist())

Nombres de las columnas:
['permit_number', 'streetname', 'Cross Street 1', 'Cross Street 2', 'Permit Type', 'Agent', 'AgentPhone', 'Permit Purpose', 'Approved Date', 'Status', 'cnn', 'permit_zipcode', 'permit_start_date', 'permit_end_date', 'permit_address', '24/7 Contact', 'Inspector', 'CurbRampWork', 'X', 'Y', 'Latitude', 'Longitude', 'bpa', 'Location', 'the_geom', 'analysis_neighborhood', 'supervisor_district', 'data_as_of', 'data_loaded_at']


## Diccionario preliminar de variables

| Nombre de la Variable | Significado |
|---|---|
| **permit_number** | Número identificador único del permiso de uso de calle. |
| **streetname** | Nombre de la calle principal donde se realiza el trabajo. |
| **Cross Street 1** | Nombre de la primera calle transversal o intersección más cercana. |
| **Cross Street 2** | Nombre de la segunda calle transversal, si aplica. |
| **Permit Type** | Categoría o tipo de permiso otorgado. |
| **Agent** | Persona, empresa o contratista responsable del permiso. |
| **AgentPhone** | Teléfono de contacto del agente o contratista responsable. |
| **Permit Purpose** | Descripción del propósito de la solicitud. |
| **Approved Date** | Fecha de aprobación oficial de la solicitud. |
| **Status** | Estado actual del permiso, por ejemplo, `ACTIVE` o `APPROVED`. |
| **cnn** | Centerline Network Number que identifica un segmento de calle. |
| **permit_zipcode** | Código postal de la ubicación del permiso. |
| **permit_start_date** | Fecha de inicio de vigencia del permiso. |
| **permit_end_date** | Fecha de término de vigencia del permiso. |
| **permit_address** | Dirección física registrada para el permiso. |
| **24/7 Contact** | Contacto de emergencia disponible durante todo el día. |
| **Inspector** | Inspector municipal asignado al permiso. |
| **CurbRampWork** | Indicador de trabajos relacionados con rampas de acera. |
| **X** | Coordenada X del sistema espacial de la ciudad. |
| **Y** | Coordenada Y del sistema espacial de la ciudad. |
| **Latitude** | Latitud geográfica de la ubicación. |
| **Longitude** | Longitud geográfica de la ubicación. |
| **bpa** | Número de solicitud de permiso de construcción asociado, si existe. |
| **Location** | Coordenada espacial utilizada para representación cartográfica. |
| **the_geom** | Geometría espacial del registro en formato de punto. |
| **analysis_neighborhood** | Barrio o zona de análisis asociada a la ubicación. |
| **supervisor_district** | Distrito del supervisor correspondiente a la ubicación. |
| **data_as_of** | Fecha hasta la cual los datos publicados estaban vigentes. |
| **data_loaded_at** | Fecha y hora en que el registro fue cargado o actualizado en la fuente. |

In [27]:
# Contamos los valores faltantes de cada columna.
print('Valores faltantes por columna:')
print(permits.isnull().sum())

Valores faltantes por columna:
permit_number               0
streetname                  0
Cross Street 1              0
Cross Street 2           1000
Permit Type                 0
Agent                       9
AgentPhone                109
Permit Purpose           1712
Approved Date              18
Status                      0
cnn                         0
permit_zipcode              0
permit_start_date         835
permit_end_date           711
permit_address           4321
24/7 Contact              202
Inspector                2505
CurbRampWork             6608
X                           8
Y                           8
Latitude                    0
Longitude                   0
bpa                      6555
Location                    0
the_geom                    0
analysis_neighborhood       3
supervisor_district         2
data_as_of                  0
data_loaded_at              0
dtype: int64


### Variables con ausencia crítica de información (Más del 60% de nulos)
Mantener variables casi vacías no aporta valor estadístico y rellenarlas inventaría datos, sesgando tu análisis.

- CurbRampWork (96.44% faltante): Al faltar casi en su totalidad, no permite establecer ningún patrón generalizable sobre los permisos.

- bpa (95.67% faltante): Es un número de solicitud de construcción que casi nunca se registra. Su ausencia es tan alta que es inservible para el análisis.

- permit_address (63.06% faltante): Además de faltar en más de la mitad de los registros, es una variable de texto libre. Para el análisis territorial, ya cuentas con alternativas mucho más limpias y completas (como el Código Postal, Latitud, Longitud y Barrio).

### Identificadores únicos y datos de contacto (Alta cardinalidad)
Estas variables son exclusivas de cada fila o son datos personales. No sirven para encontrar patrones, comparar grupos o medir duraciones.

- permit_number: Es solo el correlativo administrativo. Matemáticamente no influye en cuánto dura un permiso.

- AgentPhone y 24/7 Contact: Son números de teléfono y nombres de personas. No tienen ningún poder explicativo sobre la duración de las obras y solo añaden "ruido" al dataset.

- Agent: Aunque identifica al solicitante, indicaste que tiene 704 valores únicos (alta cardinalidad). Analizar 704 categorías distintas en un dataset de 6.800 filas fragmentará demasiado los datos, por lo que no es útil para un análisis descriptivo general.

### Variables geoespaciales redundantes
Tener la misma información repetida sobrecarga el modelo y genera redundancia.

- X e Y: Son coordenadas en un sistema de proyección local.

- Location: Es la combinación en texto de la latitud y longitud.

¿Por qué eliminarlas? Porque tu dataset ya tiene las columnas Latitude y Longitude (con 0 faltantes), que son el estándar universal para cualquier visualización territorial o mapa. Las otras tres simplemente sobran.

### Metadatos del sistema informático
- data_as_of y data_loaded_at: Indican la fecha y hora en la que el servidor municipal exportó o cargó la base de datos.

¿Por qué eliminarlas? Porque no tienen ninguna relación con el evento físico en la calle. No afectan la duración del permiso ni su ubicación.

### Criterios de exclusión de variables
En alineación con la pregunta de investigación, se determinó excluir un subconjunto de variables que no aportan valor descriptivo al cálculo de la duración o a la caracterización territorial. 

- Se descartaron variables con una ausencia crítica de datos (CurbRampWork, bpa, permit_address), las cuales superan el 60% de valores nulos. 
- Se eliminaron identificadores administrativos de alta cardinalidad y datos de contacto (permit_number, AgentPhone, 24/7 Contact), dado que no permiten la agrupación o comparación de patrones.
- Se excluyeron metadatos de exportación del sistema (data_as_of, data_loaded_at) y variables espaciales redundantes (X, Y, Location), priorizando el uso exclusivo de Latitud y Longitud para el análisis geográfico.

In [28]:
# Resumimos las estadísticas descriptivas de las columnas numéricas originales.
print('Estadísticas descriptivas de las variables numéricas:')
display(permits.describe())

Estadísticas descriptivas de las variables numéricas:


,cnn,supervisor_district
count,6.852000e+03,6850.000000
mean,9.631995e+06,5.756934
std,8.045358e+06,3.032223
min,1.000000e+05,1.000000
25%,4.079000e+06,3.000000
50%,8.004000e+06,6.000000
75%,1.204500e+07,8.000000
max,5.438300e+07,11.000000


## 2. Exploración inicial y calidad

In [29]:
# Contamos los valores faltantes por columna.
missing_by_column = permits.isna().sum()

In [30]:
# Convertimos el conteo en una tabla y calculamos el porcentaje de ausencia.
quality_summary = (
    missing_by_column
    .rename('valores_faltantes')
    .to_frame()
    .assign(porcentaje=lambda frame: (100 * frame['valores_faltantes'] / len(permits)).round(2))
    .sort_values('porcentaje', ascending=False)
)

# Mostramos las columnas con mayor proporción de datos faltantes.
quality_summary.head(12)

,valores_faltantes,porcentaje
CurbRampWork,6608,96.44
bpa,6555,95.67
permit_address,4321,63.06
Inspector,2505,36.56
Permit Purpose,1712,24.99
Cross Street 2,1000,14.59
permit_start_date,835,12.19
permit_end_date,711,10.38
24/7 Contact,202,2.95
AgentPhone,109,1.59


## Decisiones frente a los valores faltantes

- Las fechas de inicio y término no se rellenan con valores inventados, porque alterarían la duración real del permiso.
- Para analizar duración, se trabajará con registros que tengan ambas fechas válidas.
- Las variables categóricas seleccionadas se normalizan y sus ausencias se identifican como `No informado`.
- Las variables con ausencia muy alta, como `CurbRampWork` y `bpa`, se conservan en la tabla original, pero no se usarán como variables principales mientras no exista una justificación adicional.
- Las coordenadas faltantes o fuera de rango se convierten en valores ausentes y se excluyen solo de análisis geográficos.
- Solo se eliminan duplicados exactos; los permisos repetidos en varias calles se mantienen porque representan registros territoriales distintos.

## Criterio general para tratar datos faltantes

La decisión depende de tres elementos: el tipo de dato, la proporción de valores faltantes y la importancia de la variable para el análisis.

| Situación | Tratamiento recomendado |
|---|---|
| Variable numérica con pocos faltantes | Imputar con media o mediana. La mediana es preferible si existen valores extremos. |
| Variable numérica con distribución aproximadamente simétrica | Puede utilizarse la media. |
| Variable numérica sesgada o con valores atípicos | Utilizar la mediana para evitar distorsiones. |
| Variable categórica | Utilizar la moda solo si tiene sentido; alternativamente, usar No informado. |
| Fecha necesaria para calcular una duración | No inventar fechas. Excluir el registro únicamente de ese análisis. |
| Identificador o código | No reemplazar por media, mediana o moda. Mantener como texto o dejar faltante. |
| Columna con ausencia muy alta y sin utilidad analítica | Evaluar excluirla del análisis, conservando la fuente original. |
| Registro con muchos faltantes y poca relevancia | Evaluar eliminarlo, justificando la decisión. |

En este proyecto, las fechas de inicio y término son variables críticas para calcular la duración, por lo que no se imputan automáticamente. En cambio, las variables categóricas seleccionadas pueden representarse como No informado. Las decisiones se aplican sobre copias de trabajo para conservar intacto el archivo original.

In [32]:
# Contamos categorías y claves distintas sin eliminar registros territoriales.
# Un mismo permit_number puede aparecer en varias calles o tramos.
categorical_summary = pd.DataFrame({
    'estados': [permits['Status'].nunique(dropna=True)],
    'tipos_permiso': [permits['Permit Type'].nunique(dropna=True)],
    'barrios': [permits['analysis_neighborhood'].nunique(dropna=True)],
    'permisos_unicos': [permits['permit_number'].nunique(dropna=True)],
    'duplicados_exactos': [int(permits.duplicated().sum())],
})

# Presentamos los indicadores para documentar la exploración inicial.
categorical_summary

,estados,tipos_permiso,barrios,permisos_unicos,duplicados_exactos
0,2,26,41,2851,0


### Análisis de los Resultados

# duplicados_exactos (0):

- Qué significa: No hay ninguna fila que sea una copia idéntica de otra.

- Impacto analítico: Confirma la integridad básica del conjunto de datos. No necesita aplicar comandos para borrar filas duplicadas, lo que simplifica la limpieza de datos.

# estados (2):

- Qué significa: Solo existen dos categorías de estado para los permisos (según vimos antes, suelen ser "Active" y "Approved").

- Impacto analítico: Es una variable de muy baja cardinalidad. Permitirá hacer una comparación muy limpia y binaria en los gráficos.

# tipos_permiso (26):

- Qué significa: Hay 26 naturalezas distintas de permisos en la ciudad.

- Impacto analítico: Es un número perfecto. Tiene suficiente variedad para que descubrir cosas interesantes (responder qué tipo de permiso demora más), pero no son tantas categorías como para que un gráfico de barras se vuelva ilegible.

# barrios (41):

- Qué significa: La base de datos tiene cobertura sobre 41 zonas o vecindarios de San Francisco.

- Impacto analítico: Responde directamente a la "ubicación territorial" de la pregunta de investigación. Se prodra agrupar los datos para ver en qué barrio se concentran las obras más largas.

## 3. Transformación de fechas, duración y coordenadas

Las fechas se convierten con errors='coerce': un valor ilegible se transforma en faltante y no se inventa información. La duración se expresa en días y solo se acepta cuando es mayor o igual que cero.

In [5]:
# Definimos las columnas que contienen fechas en formato mes/día/año.
date_columns = ['Approved Date', 'permit_start_date', 'permit_end_date']

In [6]:
# Convertimos texto a fechas; los valores ilegibles se transforman en NaT.
for column in date_columns:
    permits[column] = pd.to_datetime(
        permits[column],
        errors='coerce',
        format='mixed',
    )

In [39]:
# Calculamos la duración declarada del permiso en días.
permits['duration_days'] = (
    permits['permit_end_date'] - permits['permit_start_date']
).dt.days

# Marcamos como faltantes las duraciones negativas o no calculables.
invalid_duration = permits['duration_days'].lt(0).fillna(False)
permits.loc[invalid_duration, 'duration_days'] = np.nan

In [40]:
# Convertimos las coordenadas a numéricas y anulamos valores fuera de rango.
coordinate_bounds = {'Latitude': (-90, 90), 'Longitude': (-180, 180)}
for column, (lower_bound, upper_bound) in coordinate_bounds.items():
    values = pd.to_numeric(
        permits[column].astype('string').str.replace(',', '.', regex=False).str.strip(),
        errors='coerce',
    )
    permits[column] = values.where(values.between(lower_bound, upper_bound))

# Resumimos estadísticamente las duraciones que pudieron calcularse.
duration_summary = permits['duration_days'].describe()
duration_summary

count    6014.000000
mean     1050.156302
std      1421.632068
min         0.000000
25%        91.000000
50%       478.000000
75%       785.000000
max      7884.000000
Name: duration_days, dtype: float64

### Análisis de los Resultados
# count (6.014):

- Qué significa: Se logró calcular la duración para 6.014 registros.

- Impacto analítico: Si el dataset original tenía 6.852 filas, se pierde alrededor de 838 registros al hacer este cálculo. Esto concuerda perfectamente con el diagnóstico anterior, donde se ve que las fechas de inicio y término tenían valores nulos. Mantiene el ~87% de los datos, lo cual es excelente y representativo.

# mean (1.050 días) vs. 50% (Mediana: 478 días):

- Qué significa: El promedio matemático es de 1.050 días (casi 3 años), pero la mediana (el valor que está exactamente en el medio) es de 478 días (poco más de 1 año).

- Impacto analítico: Cuando el promedio es tan superior a la mediana, significa que la distribución tiene un sesgo hacia la derecha (asimetría positiva). Es decir, la mayoría de los permisos duran alrededor de un año, pero hay unos pocos permisos extremadamente largos que "tiran" del promedio hacia arriba. En en análisis, la mediana será un indicador mucho más confiable que el promedio.

# min (0 días):

- Qué significa: Hay permisos que empiezan y terminan el mismo día (trabajos exprés o de emergencia).

# max (7.884 días):

- Qué significa: El permiso más largo dura más de 21 años (7.884 / 365)

-  Impacto analítico: Esto es una alerta gigante de valores atípicos (outliers). Podría ser un error de digitación en la plataforma de la ciudad (alguien puso año 2045 en vez de 2025) o quizás un mega-proyecto de infraestructura a larguísimo plazo.

# std (1.421 días):

- Qué significa: La desviación estándar es altísima (mayor que el promedio).

- Impacto analítico: Confirma la presencia de una dispersión extrema y outliers severos.